<a href="https://colab.research.google.com/github/mmorari-cmyk/EstructuraDeDatos/blob/main/TcPROYEECTO_ESTRUCTURAS_DE_DATOS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# CELDA 1 - CONFIGURACIÓN Y CLASES
# ============================================================

import random
from statistics import mean


# ------------------------------------------------------------
# CONFIGURACIÓN GENERAL
# ------------------------------------------------------------

# Permite repetir resultados.
random.seed(42)

# Número de cajeros.
NUMERO_CAJEROS = 3

# Tiempo total: 8 horas.
TIEMPO_SIMULACION = 28800

# Número de simulaciones.
NUMERO_SIMULACIONES = 10


# ------------------------------------------------------------
# PARÁMETROS DEL ARTÍCULO
# ------------------------------------------------------------

# Media entre llegadas: 1.5 min.
MEDIA_LLEGADA = 90

# Media atención: 3.54 min.
MEDIA_ATENCION = 212


# ============================================================
# CLASE CLIENTE
# ============================================================

class Cliente:

    def __init__(self, id_cliente, tiempo_llegada):

        # ID cliente.
        self.id = id_cliente

        # Momento llegada.
        self.tiempo_llegada = tiempo_llegada

        # Tiempo atención exponencial.
        self.tiempo_atencion = max(
            1,
            int(random.expovariate(1 / MEDIA_ATENCION))
        )

        # Tiempo espera.
        self.tiempo_espera = 0


# ============================================================
# CLASE CAJERO
# ============================================================

class Cajero:

    def __init__(self, id_cajero):

        # ID cajero.
        self.id = id_cajero

        # Cliente actual.
        self.cliente_actual = None

        # Tiempo restante.
        self.tiempo_restante = 0

        # Tiempo ocupado.
        self.tiempo_ocupado = 0


    # Verifica disponibilidad.
    def esta_libre(self):

        return self.cliente_actual is None


    # Asigna cliente.
    def asignar_cliente(self, cliente):

        self.cliente_actual = cliente
        self.tiempo_restante = cliente.tiempo_atencion


    # Actualiza estado.
    def actualizar(self):

        if self.cliente_actual:

            self.tiempo_restante -= 1
            self.tiempo_ocupado += 1

            # Finaliza atención.
            if self.tiempo_restante <= 0:

                self.cliente_actual = None


# ============================================================
# CLASE COLA FIFO
# ============================================================

class Cola:

    def __init__(self):

        # Lista clientes.
        self.items = []


    # Agrega cliente.
    def encolar(self, cliente):

        self.items.append(cliente)


    # Retira cliente.
    def desencolar(self):

        if not self.esta_vacia():

            return self.items.pop(0)


    # Verifica cola vacía.
    def esta_vacia(self):

        return len(self.items) == 0


    # Tamaño cola.
    def tamano(self):

        return len(self.items)

In [2]:
# ============================================================
# CELDA 2 - FUNCIONES DE SIMULACIÓN
# ============================================================


# ============================================================
# FUNCIÓN: SIMULACIÓN COLA ÚNICA
# ============================================================

def simulacion_cola_unica():
    """
    Simula una cola única compartida por todos los cajeros.
    Los clientes se asignan al primer cajero disponible.
    Retorna métricas: espera, atendidos, restantes, utilización.
    """

    # Cajeros - se crean nuevos en cada simulación.
    cajeros = [Cajero(i) for i in range(NUMERO_CAJEROS)]

    # Cola única compartida.
    cola = Cola()

    # Contador de clientes generados.
    id_cliente = 0

    # Tiempo de la próxima llegada (en segundos).
    tiempo_proxima_llegada = int(random.expovariate(1 / MEDIA_LLEGADA))

    # Métricas acumuladas.
    tiempos_espera = []
    clientes_atendidos = 0

    tiempos_servicio = []

    # ------------------------------------------------------------
    # BUCLE PRINCIPAL - segundo a segundo
    # ------------------------------------------------------------

    for tiempo_actual in range(TIEMPO_SIMULACION):

        # --------------------------------------------------------
        # LLEGADA DE CLIENTES
        # While permite múltiples llegadas en el mismo segundo.
        # --------------------------------------------------------

        while tiempo_actual >= tiempo_proxima_llegada:

            # Crear nuevo cliente con tiempo de llegada real.
            nuevo_cliente = Cliente(id_cliente, tiempo_proxima_llegada)
            id_cliente += 1

            # Encolar cliente.
            cola.encolar(nuevo_cliente)

            # Programar PRÓXIMA llegada acumulando el intervalo.
            tiempo_proxima_llegada += max(
                1,
                int(random.expovariate(1 / MEDIA_LLEGADA))
            )

        # --------------------------------------------------------
        # ACTUALIZAR CAJEROS y contar clientes que terminaron.
        # --------------------------------------------------------

        for cajero in cajeros:

            if not cajero.esta_libre():

                # Guardar estado antes de actualizar.
                estaba_ocupado = True

                cajero.actualizar()

                # Si ahora está libre → terminó de atender.
                if cajero.esta_libre():
                    clientes_atendidos += 1

        # --------------------------------------------------------
        # ASIGNACIÓN: cajero libre → siguiente cliente en cola.
        # --------------------------------------------------------

        for cajero in cajeros:

            if cajero.esta_libre() and not cola.esta_vacia():

                # Retirar cliente de la cola.
                cliente = cola.desencolar()

                # Tiempo de espera en segundos → se convierte a min al final.
                cliente.tiempo_espera = tiempo_actual - cliente.tiempo_llegada
                tiempos_espera.append(cliente.tiempo_espera)

                # Asignar cajero.
                cajero.asignar_cliente(cliente)

    # ------------------------------------------------------------
    # MÉTRICAS FINALES
    # ------------------------------------------------------------

    # Espera promedio en minutos (tiempos_espera está en segundos).
    espera_promedio = mean(tiempos_espera) / 60 if tiempos_espera else 0

    # Clientes que quedaron en cola sin ser atendidos.
    clientes_restantes = cola.tamano()

    # Utilización promedio de cajeros (%).
    utilizacion = mean(
        (c.tiempo_ocupado / TIEMPO_SIMULACION) * 100
        for c in cajeros
    )

    return {
        "espera": espera_promedio,
        "atendidos": clientes_atendidos,
        "restantes": clientes_restantes,
        "utilizacion": utilizacion
    }


# ============================================================
# FUNCIÓN: SIMULACIÓN TRES COLAS INDEPENDIENTES
# ============================================================

def simulacion_tres_colas():
    """
    Simula tres colas independientes, una por cajero.
    Cada cliente se asigna a la cola con menos personas.
    Retorna métricas: espera, atendidos, restantes, utilización.
    """

    # Cajeros - se crean nuevos en cada simulación.
    cajeros = [Cajero(i) for i in range(NUMERO_CAJEROS)]

    # Una cola independiente por cajero.
    colas = [Cola() for _ in range(NUMERO_CAJEROS)]

    # Contador de clientes generados.
    id_cliente = 0

    # Tiempo de la próxima llegada (en segundos).
    tiempo_proxima_llegada = int(random.expovariate(1 / MEDIA_LLEGADA))

    # Métricas acumuladas.
    tiempos_espera = []
    clientes_atendidos = 0

    tiempos_servicio = []

    # ------------------------------------------------------------
    # BUCLE PRINCIPAL - segundo a segundo
    # ------------------------------------------------------------

    for tiempo_actual in range(TIEMPO_SIMULACION):

        # --------------------------------------------------------
        # LLEGADA DE CLIENTES
        # While permite múltiples llegadas en el mismo segundo.
        # --------------------------------------------------------

        while tiempo_actual >= tiempo_proxima_llegada:

            # Crear nuevo cliente con tiempo de llegada real.
            nuevo_cliente = Cliente(id_cliente, tiempo_proxima_llegada)
            id_cliente += 1

            # Asignar a la cola con menor número de personas.
            # Si hay empate, elige la de menor índice.
            indice_menor = min(
                range(NUMERO_CAJEROS),
                key=lambda i: colas[i].tamano()
            )
            colas[indice_menor].encolar(nuevo_cliente)

            # Programar PRÓXIMA llegada acumulando el intervalo.
            tiempo_proxima_llegada += max(
                1,
                int(random.expovariate(1 / MEDIA_LLEGADA))
            )

        # --------------------------------------------------------
        # ACTUALIZAR CAJEROS y contar clientes que terminaron.
        # --------------------------------------------------------

        for cajero in cajeros:

            if not cajero.esta_libre():

                cajero.actualizar()

                # Si ahora está libre → terminó de atender.
                if cajero.esta_libre():
                    clientes_atendidos += 1

        # --------------------------------------------------------
        # ASIGNACIÓN: cada cajero atiende su propia cola.
        # --------------------------------------------------------

        for i, cajero in enumerate(cajeros):

            if cajero.esta_libre() and not colas[i].esta_vacia():

                # Retirar cliente de la cola correspondiente.
                cliente = colas[i].desencolar()

                # Tiempo de espera en segundos → se convierte a min al final.
                cliente.tiempo_espera = tiempo_actual - cliente.tiempo_llegada
                tiempos_espera.append(cliente.tiempo_espera)

                # Asignar cajero.
                cajero.asignar_cliente(cliente)

    # ------------------------------------------------------------
    # MÉTRICAS FINALES
    # ------------------------------------------------------------

    # Espera promedio en minutos (tiempos_espera está en segundos).
    espera_promedio = mean(tiempos_espera) / 60 if tiempos_espera else 0

    # Clientes que quedaron en colas sin ser atendidos.
    clientes_restantes = sum(c.tamano() for c in colas)

    # Utilización promedio de cajeros (%).
    utilizacion = mean(
        (c.tiempo_ocupado / TIEMPO_SIMULACION) * 100
        for c in cajeros
    )

    return {
        "espera": espera_promedio,
        "atendidos": clientes_atendidos,
        "restantes": clientes_restantes,
        "utilizacion": utilizacion
    }

In [6]:
# ============================================================
# CELDA 2 - FUNCIONES Y SIMULACIONES
# ============================================================

# ------------------------------------------------------------
# GENERAR LLEGADA
# ------------------------------------------------------------

# Genera llegadas exponenciales.

def generar_proxima_llegada(tiempo_actual):

    intervalo = max(
        1,
        int(random.expovariate(1 / MEDIA_LLEGADA))
    )

    return tiempo_actual + intervalo


# ============================================================
# COLA ÚNICA
# ============================================================

def simulacion_cola_unica():

    # Cola principal.
    cola_general = Cola()

    # Crear cajeros.
    cajeros = [Cajero(i + 1) for i in range(NUMERO_CAJEROS)]

    # Métricas.
    tiempos_espera = []
    clientes_atendidos = 0
    tiempos_servicio = [] # Inicializar tiempos_servicio

    id_cliente = 1
    proxima_llegada = generar_proxima_llegada(0)


    # CICLO PRINCIPAL

    for tiempo_actual in range(TIEMPO_SIMULACION):


        # NUEVO CLIENTE

        if tiempo_actual >= proxima_llegada:

            cliente = Cliente(id_cliente, tiempo_actual) # Crear cliente primero
            tiempos_servicio.append(cliente.tiempo_atencion) # Luego añadir el tiempo de servicio

            cola_general.encolar(cliente)

            id_cliente += 1

            proxima_llegada = generar_proxima_llegada(tiempo_actual)


        # REVISAR CAJEROS

        for cajero in cajeros:

            if cajero.esta_libre() and not cola_general.esta_vacia():

                cliente = cola_general.desencolar()

                # Tiempo espera.
                cliente.tiempo_espera = tiempo_actual - cliente.tiempo_llegada

                tiempos_espera.append(cliente.tiempo_espera)

                cajero.asignar_cliente(cliente)


            antes = cajero.cliente_actual

            cajero.actualizar()


            # Cliente atendido.
            if antes and cajero.cliente_actual is None:

                clientes_atendidos += 1


    # Clientes restantes.
    restantes = cola_general.tamano() + sum(
        not cajero.esta_libre()
        for cajero in cajeros
    )


    # Utilización promedio.
    utilizacion = mean([
        (c.tiempo_ocupado / TIEMPO_SIMULACION) * 100
        for c in cajeros
    ])


    # Resultados finales.
    return {

        "espera": mean(tiempos_espera) / 60 if tiempos_espera else 0,

        "servicio": mean(tiempos_servicio) / 60 if tiempos_servicio else 0,

        "atendidos": clientes_atendidos,

        "restantes": restantes,

        "utilizacion": utilizacion
    }


# ============================================================
# TRES COLAS
# ============================================================

def simulacion_tres_colas():

    # Crear colas.
    colas = [Cola() for _ in range(NUMERO_CAJEROS)]

    # Crear cajeros.
    cajeros = [Cajero(i + 1) for i in range(NUMERO_CAJEROS)]

    # Métricas.
    tiempos_espera = []
    clientes_atendidos = 0
    tiempos_servicio = [] # Inicializar tiempos_servicio

    id_cliente = 1
    proxima_llegada = generar_proxima_llegada(0)


    # CICLO PRINCIPAL

    for tiempo_actual in range(TIEMPO_SIMULACION):


        # NUEVO CLIENTE

        if tiempo_actual >= proxima_llegada:

            cliente = Cliente(id_cliente, tiempo_actual)
            tiempos_servicio.append(cliente.tiempo_atencion) # Añadir el tiempo de servicio

            # Cola más corta.
            cola_menor = min(colas, key=lambda c: c.tamano())

            cola_menor.encolar(cliente)

            id_cliente += 1

            proxima_llegada = generar_proxima_llegada(tiempo_actual)


        # REVISAR CAJEROS

        for i in range(NUMERO_CAJEROS):

            cajero = cajeros[i]
            cola = colas[i]


            if cajero.esta_libre() and not cola.esta_vacia():

                cliente = cola.desencolar()

                # Tiempo espera.
                cliente.tiempo_espera = tiempo_actual - cliente.tiempo_llegada

                tiempos_espera.append(cliente.tiempo_espera)

                cajero.asignar_cliente(cliente)


            antes = cajero.cliente_actual

            cajero.actualizar()


            # Cliente atendido.
            if antes and cajero.cliente_actual is None:

                clientes_atendidos += 1


    # Clientes restantes.
    restantes = sum(c.tamano() for c in colas) + sum(
        not cajero.esta_libre()
        for cajero in cajeros
    )


    # Utilización promedio.
    utilizacion = mean([
        (c.tiempo_ocupado / TIEMPO_SIMULACION) * 100
        for c in cajeros
    ])


    # Resultados finales.
    return {

        "espera": mean(tiempos_espera) / 60 if tiempos_espera else 0,

        "servicio": mean(tiempos_servicio) / 60 if tiempos_servicio else 0,

        "atendidos": clientes_atendidos,

        "restantes": restantes,

        "utilizacion": utilizacion
    }

In [7]:
# ============================================================
# CELDA 3 - EJECUCIÓN Y RESULTADOS
# ============================================================

# ------------------------------------------------------------
# LISTAS RESULTADOS
# ------------------------------------------------------------

resultados_unica = []
resultados_tres = []


# ============================================================
# EJECUTAR SIMULACIONES
# ============================================================

# Ejecutar múltiples pruebas.

for _ in range(NUMERO_SIMULACIONES):

    resultados_unica.append(simulacion_cola_unica())

    resultados_tres.append(simulacion_tres_colas())


# ============================================================
# PROMEDIOS FINALES
# ============================================================

# Espera promedio.
promedio_unica = mean(r["espera"] for r in resultados_unica)
promedio_tres = mean(r["espera"] for r in resultados_tres)

#Tiempo promedio de servicio
servicio_unica = mean(r["servicio"] for r in resultados_unica)
servicio_tres = mean(r["servicio"] for r in resultados_tres)

# Clientes atendidos.
atendidos_unica = mean(r["atendidos"] for r in resultados_unica)
atendidos_tres = mean(r["atendidos"] for r in resultados_tres)

# Clientes restantes.
restantes_unica = mean(r["restantes"] for r in resultados_unica)
restantes_tres = mean(r["restantes"] for r in resultados_tres)

# Utilización promedio.
utilizacion_unica = mean(r["utilizacion"] for r in resultados_unica)
utilizacion_tres = mean(r["utilizacion"] for r in resultados_tres)


# ============================================================
# CAPACIDAD OPERATIVA
# ============================================================

# Clientes por hora.
atencion_total_unica = atendidos_unica / 8
atencion_total_tres = atendidos_tres / 8

# Clientes por cajero.
atencion_cajero_unica = atencion_total_unica / NUMERO_CAJEROS
atencion_cajero_tres = atencion_total_tres / NUMERO_CAJEROS


# ============================================================
# MOSTRAR RESULTADOS
# ============================================================

print("====================================")
print("RESULTADOS FINALES")
print("====================================")


# COLA ÚNICA

print("\nCOLA ÚNICA")

print(f"Espera promedio: {promedio_unica:.2f} minutos")
print(f"Tiempo promedio servicio: {servicio_unica:.2f} minutos")

print(f"Clientes atendidos promedio: {atendidos_unica:.0f}")

print(f"Clientes restantes promedio: {restantes_unica:.0f}")

print(f"Utilización promedio cajeros: {utilizacion_unica:.2f}%")

print("\nCapacidad operativa:")

print(f"Atención total del sistema: {atencion_total_unica:.0f} clientes/hora")

print(f"Atención promedio por cajero: {atencion_cajero_unica:.0f} clientes/hora")


# TRES COLAS

print("\nTRES COLAS")

print(f"Espera promedio: {promedio_tres:.2f} minutos")

print(f"Tiempo promedio servicio: {servicio_tres:.2f} minutos")

print(f"Clientes atendidos promedio: {atendidos_tres:.0f}")

print(f"Clientes restantes promedio: {restantes_tres:.0f}")

print(f"Utilización promedio cajeros: {utilizacion_tres:.2f}%")

print("\nCapacidad operativa:")

print(f"Atención total del sistema: {atencion_total_tres:.0f} clientes/hora")

print(f"Atención promedio por cajero: {atencion_cajero_tres:.0f} clientes/hora")


# ============================================================
# CONCLUSIÓN
# ============================================================

if promedio_unica < promedio_tres:

    print("\nConclusión: La cola única fue más eficiente.")

else:

    print("\nConclusión: Las tres colas fueron más eficientes.")

RESULTADOS FINALES

COLA ÚNICA
Espera promedio: 2.96 minutos
Tiempo promedio servicio: 3.63 minutos
Clientes atendidos promedio: 317
Clientes restantes promedio: 5
Utilización promedio cajeros: 79.74%

Capacidad operativa:
Atención total del sistema: 40 clientes/hora
Atención promedio por cajero: 13 clientes/hora

TRES COLAS
Espera promedio: 5.16 minutos
Tiempo promedio servicio: 3.53 minutos
Clientes atendidos promedio: 325
Clientes restantes promedio: 4
Utilización promedio cajeros: 79.48%

Capacidad operativa:
Atención total del sistema: 41 clientes/hora
Atención promedio por cajero: 14 clientes/hora

Conclusión: La cola única fue más eficiente.
